In [2]:
from pathlib import Path
import json
import pandas as pd

BANDIT_RESULTS_DIR = Path("bandit-results")

records = []

for issue_file in BANDIT_RESULTS_DIR.glob("*/*/introduced_issues.json"):
    # 路径类似:
    # bandit-results/owner__repo/pr_123/introduced_issues.json
    repo_dir = issue_file.parent.parent.name
    pr_dir = issue_file.parent.name

    owner_repo = repo_dir.split("__", 1)
    if len(owner_repo) == 2:
        owner, repo = owner_repo
    else:
        owner, repo = None, repo_dir

    pr_number = pr_dir.replace("pr_", "")

    with open(issue_file, "r", encoding="utf-8") as f:
        issues = json.load(f)

    records.append({
        "owner": owner,
        "repo": repo,
        "pr_number": pr_number,
        "issue_file": str(issue_file),
        "introduced_issue_count": len(issues),
        "issues": issues,
    })

pr_df = pd.DataFrame(records)
pr_df = pr_df.sort_values(["introduced_issue_count", "owner", "repo"], ascending=[False, True, True]).reset_index(drop=True)

pr_df.head()

,owner,repo,pr_number,issue_file,introduced_issue_count,issues
0,mlflow,mlflow,16112,bandit-results/mlflow__mlflow/pr_16112/introdu...,376,[{'code': '8 import json 9 import subprocess 1...
1,quantalogic,quantalogic,64,bandit-results/quantalogic__quantalogic/pr_64/...,324,[{'code': '146 # Test that key fun...
2,willccbb,verifiers,124,bandit-results/willccbb__verifiers/pr_124/intr...,260,[{'code': '64 65 assert len(r...
3,marimo-team,marimo,4010,bandit-results/marimo-team__marimo/pr_4010/int...,246,[{'code': '22 model = simple(lambda x: x *...
4,willccbb,verifiers,123,bandit-results/willccbb__verifiers/pr_123/intr...,219,[{'code': '39 ) 40 assert env....


In [3]:
issue_rows = []

for _, row in pr_df.iterrows():
    for issue in row["issues"]:
        issue_rows.append({
            "owner": row["owner"],
            "repo": row["repo"],
            "pr_number": row["pr_number"],
            "filename": issue.get("filename"),
            "test_id": issue.get("test_id"),
            "issue_text": issue.get("issue_text"),
            "line_number": issue.get("line_number"),
            "issue_severity": issue.get("issue_severity"),
            "issue_confidence": issue.get("issue_confidence"),
            "issue_cwe": (issue.get("issue_cwe") or {}).get("id"),
            "more_info": issue.get("more_info"),
            "code": issue.get("code"),
        })

issue_df = pd.DataFrame(issue_rows)
issue_df = issue_df.sort_values(
    ["owner", "repo", "pr_number", "filename", "line_number"],
    ascending=True
).reset_index(drop=True)


issue_df = issue_df[issue_df["test_id"].notna()].reset_index(drop=True)
issue_df.head()

,owner,repo,pr_number,filename,test_id,issue_text,line_number,issue_severity,issue_confidence,issue_cwe,more_info,code
0,567-labs,instructor,1567,./tests/test_dynamic_model_creation.py,B101,Use of assert detected. The enclosed code will...,33,LOW,HIGH,703,https://bandit.readthedocs.io/en/1.9.2/plugins...,32 \n33 assert schema['properties']['n...
1,567-labs,instructor,1567,./tests/test_dynamic_model_creation.py,B101,Use of assert detected. The enclosed code will...,34,LOW,HIGH,703,https://bandit.readthedocs.io/en/1.9.2/plugins...,33 assert schema['properties']['name']['de...
2,567-labs,instructor,1567,./tests/test_dynamic_model_creation.py,B101,Use of assert detected. The enclosed code will...,35,LOW,HIGH,703,https://bandit.readthedocs.io/en/1.9.2/plugins...,34 assert schema['properties']['age']['des...
3,567-labs,instructor,1567,./tests/test_dynamic_model_creation.py,B101,Use of assert detected. The enclosed code will...,37,LOW,HIGH,703,https://bandit.readthedocs.io/en/1.9.2/plugins...,36 \n37 assert 'default' not in schema...
4,567-labs,instructor,1567,./tests/test_dynamic_model_creation.py,B101,Use of assert detected. The enclosed code will...,38,LOW,HIGH,703,https://bandit.readthedocs.io/en/1.9.2/plugins...,37 assert 'default' not in schema['propert...


In [4]:
test_files_df = issue_df[
    issue_df["filename"].str.contains("test", case=False, na=False)
]

test_files_df

,owner,repo,pr_number,filename,test_id,issue_text,line_number,issue_severity,issue_confidence,issue_cwe,more_info,code
0,567-labs,instructor,1567,./tests/test_dynamic_model_creation.py,B101,Use of assert detected. The enclosed code will...,33,LOW,HIGH,703,https://bandit.readthedocs.io/en/1.9.2/plugins...,32 \n33 assert schema['properties']['n...
1,567-labs,instructor,1567,./tests/test_dynamic_model_creation.py,B101,Use of assert detected. The enclosed code will...,34,LOW,HIGH,703,https://bandit.readthedocs.io/en/1.9.2/plugins...,33 assert schema['properties']['name']['de...
2,567-labs,instructor,1567,./tests/test_dynamic_model_creation.py,B101,Use of assert detected. The enclosed code will...,35,LOW,HIGH,703,https://bandit.readthedocs.io/en/1.9.2/plugins...,34 assert schema['properties']['age']['des...
3,567-labs,instructor,1567,./tests/test_dynamic_model_creation.py,B101,Use of assert detected. The enclosed code will...,37,LOW,HIGH,703,https://bandit.readthedocs.io/en/1.9.2/plugins...,36 \n37 assert 'default' not in schema...
4,567-labs,instructor,1567,./tests/test_dynamic_model_creation.py,B101,Use of assert detected. The enclosed code will...,38,LOW,HIGH,703,https://bandit.readthedocs.io/en/1.9.2/plugins...,37 assert 'default' not in schema['propert...
...,...,...,...,...,...,...,...,...,...,...,...,...
9389,zfit,zfit,644,./tests/model/test_positive_pdf.py,B101,Use of assert detected. The enclosed code will...,212,LOW,HIGH,703,https://bandit.readthedocs.io/en/1.9.2/plugins...,211 positive_ext_vals = positive_ext_poly....
9390,zfit,zfit,644,./tests/model/test_positive_pdf.py,B101,Use of assert detected. The enclosed code will...,234,LOW,HIGH,703,https://bandit.readthedocs.io/en/1.9.2/plugins...,"233 # Should inherit obs, norm, and extend..."
9391,zfit,zfit,644,./tests/model/test_positive_pdf.py,B101,Use of assert detected. The enclosed code will...,235,LOW,HIGH,703,https://bandit.readthedocs.io/en/1.9.2/plugins...,234 assert positive.obs == extended_gauss....
9392,zfit,zfit,644,./tests/model/test_positive_pdf.py,B101,Use of assert detected. The enclosed code will...,236,LOW,HIGH,703,https://bandit.readthedocs.io/en/1.9.2/plugins...,235 assert positive.norm == extended_gauss...


In [5]:
issue_df_clean = issue_df[
    ~issue_df["filename"].str.contains("test", case=False, na=False)
].copy()

issue_df_clean

,owner,repo,pr_number,filename,test_id,issue_text,line_number,issue_severity,issue_confidence,issue_cwe,more_info,code
48,AgentOps-AI,agentops,1064,./agentops/instrumentation/agno/instrumentor.py,B110,"Try, Except, Pass detected.",533,LOW,HIGH,703,https://bandit.readthedocs.io/en/1.9.2/plugins...,"532 parent_context, parent..."
49,AgentOps-AI,agentops,1064,./agentops/instrumentation/agno/instrumentor.py,B110,"Try, Except, Pass detected.",548,LOW,HIGH,703,https://bandit.readthedocs.io/en/1.9.2/plugins...,"547 parent_context, pa..."
50,AgentOps-AI,agentops,1064,./agentops/instrumentation/smolagents/attribut...,B110,"Try, Except, Pass detected.",87,LOW,HIGH,703,https://bandit.readthedocs.io/en/1.9.2/plugins...,86 \n87 except Exception:\n88 # If...
51,AgentOps-AI,agentops,1064,./agentops/instrumentation/smolagents/attribut...,B110,"Try, Except, Pass detected.",132,LOW,HIGH,703,https://bandit.readthedocs.io/en/1.9.2/plugins...,131 \n132 except Exception:\n133 #...
52,AgentOps-AI,agentops,1064,./agentops/instrumentation/smolagents/attribut...,B110,"Try, Except, Pass detected.",170,LOW,HIGH,703,https://bandit.readthedocs.io/en/1.9.2/plugins...,169 \n170 except Exception:\n171 #...
...,...,...,...,...,...,...,...,...,...,...,...,...
9322,willccbb,verifiers,161,./environments/toxicity_explanation/toxicity_e...,B110,"Try, Except, Pass detected.",132,LOW,HIGH,703,https://bandit.readthedocs.io/en/1.9.2/plugins...,"131 return max(0.0, min(1.0, s..."
9323,yutto-dev,yutto,539,./src/yutto/path_templates.py,B101,Use of assert detected. The enclosed code will...,100,LOW,HIGH,703,https://bandit.readthedocs.io/en/1.9.2/plugins...,99 continue\n100 assert is...
9324,yutto-dev,yutto,539,./src/yutto/types.py,B101,Use of assert detected. The enclosed code will...,87,LOW,HIGH,703,https://bandit.readthedocs.io/en/1.9.2/plugins...,86 def bv2av(bvid: str) -> int:\n87 ...
9333,zfit,zfit,644,./src/zfit/models/postprocess.py,B101,Use of assert detected. The enclosed code will...,76,LOW,HIGH,703,https://bandit.readthedocs.io/en/1.9.2/plugins...,"75 """"""Return the PDF value ensuring it..."


In [7]:
issue_df_clean['issue_cwe'].value_counts()

issue_cwe
703    599
78     271
330    191
494     45
89      33
259     27
20      26
400     24
22      20
377     18
605     13
327     11
94       4
502      2
79       1
Name: count, dtype: int64

In [9]:
total_prs = len(pr_df)
total_issues = len(issue_df)

print("Total PRs with introduced issues:", total_prs)
print("Total introduced issues:", total_issues)

Total PRs with introduced issues: 802
Total introduced issues: 9394


In [10]:
issue_df['issue_cwe'].value_counts()

issue_cwe
703    8184
78      668
330     212
259      66
377      53
494      47
89       35
400      31
20       26
22       20
605      18
327      15
502      12
94        4
732       1
79        1
295       1
Name: count, dtype: int64